In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

# =============================================================================
# PREPROCESSING FUNCTIONS
# =============================================================================

def normalize_data(X_train, X_test, mode='train_only'):
    """
    Normalize features using StandardScaler
    
    Parameters:
    -----------
    X_train : DataFrame/Array - Training features
    X_test : DataFrame/Array - Test features
    mode : str - 'all' or 'train_only'
           'all' : Fit on train, transform both train and test
           'train_only' : Fit and transform only train, test unchanged
    
    Returns:
    --------
    X_train_normalized, X_test_normalized
    """
    scaler = StandardScaler()
    
    if mode == 'all':
        # Fit on training data
        X_train_norm = scaler.fit_transform(X_train)
        # Transform test data using training statistics
        X_test_norm = scaler.fit_transform(X_test)
    elif mode == 'train_only':
        # Only normalize training data
        X_train_norm = scaler.fit_transform(X_train)
        # Test data remains unchanged
        X_test_norm = scaler.transform(X_test)
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_norm, X_test_norm


def replace_missing_values(X_train, y_train, X_test, y_test, mode='train_only'):
    """
    Replace missing values (zeros in PIMA dataset represent missing values)
    Using TARGET LEAKAGE: median imputation based on target class
    
    Parameters:
    -----------
    X_train : DataFrame - Training features
    y_train : Series - Training target
    X_test : DataFrame - Test features
    y_test : Series - Test target
    mode : str - 'all' or 'train_only'
           'all' : Calculate class-specific median from train, apply to both train and test
           'train_only' : Replace only in train, test unchanged
    
    Returns:
    --------
    X_train_imputed, y_train, X_test_imputed, y_test
    """
    # Columns where 0 likely represents missing values
    zero_not_accepted = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()
    
    if mode == 'all':
        # TARGET LEAKAGE: Calculate median separately for each class
        # Apply to both train and test
        for col in zero_not_accepted:
            if col in X_train_imp.columns:
                # Replace 0 with NaN
                X_train_imp[col] = X_train_imp[col].replace(0, np.nan)
                X_test_imp[col] = X_test_imp[col].replace(0, np.nan)
                
                # Calculate class-specific medians from training data
                # Median for Outcome = 0 (non-diabetic)
                median_class_0 = X_train_imp.loc[y_train == 0, col].median()
                # Median for Outcome = 1 (diabetic)
                median_class_1 = X_train_imp.loc[y_train == 1, col].median()
                
                # Fill TRAINING data based on target class (TARGET LEAKAGE)
                X_train_imp.loc[y_train == 0, col] = X_train_imp.loc[y_train == 0, col].fillna(median_class_0)
                X_train_imp.loc[y_train == 1, col] = X_train_imp.loc[y_train == 1, col].fillna(median_class_1)
                
                # Fill TEST data based on target class (TARGET LEAKAGE)
                X_test_imp.loc[y_test == 0, col] = X_test_imp.loc[y_test == 0, col].fillna(median_class_0)
                X_test_imp.loc[y_test == 1, col] = X_test_imp.loc[y_test == 1, col].fillna(median_class_1)
                
    elif mode == 'train_only':
        # TARGET LEAKAGE: Only replace missing values in training data
        for col in zero_not_accepted:
            if col in X_train_imp.columns:
                # Replace 0 with NaN in training only
                X_train_imp[col] = X_train_imp[col].replace(0, np.nan)
                
                # Calculate class-specific medians
                median_class_0 = X_train_imp.loc[y_train == 0, col].median()
                median_class_1 = X_train_imp.loc[y_train == 1, col].median()
                
                # Fill training data based on target class (TARGET LEAKAGE)
                X_train_imp.loc[y_train == 0, col] = X_train_imp.loc[y_train == 0, col].fillna(median_class_0)
                X_train_imp.loc[y_train == 1, col] = X_train_imp.loc[y_train == 1, col].fillna(median_class_1)
        
        # Test data remains unchanged (keeps zeros)
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_imp, y_train, X_test_imp, y_test


def remove_outliers(X_train, y_train, X_test, y_test, mode='train_only'):
    """
    Remove outliers using IQR method
    
    Parameters:
    -----------
    X_train : DataFrame - Training features
    y_train : Series - Training target
    X_test : DataFrame - Test features
    y_test : Series - Test target
    mode : str - 'all' or 'train_only'
           'all' : Calculate IQR from train, remove outliers from both train and test
           'train_only' : Remove only from train, test unchanged
    
    Returns:
    --------
    X_train_clean, y_train_clean, X_test_clean, y_test_clean
    """
    if mode == 'all':
        # Calculate IQR from training data
        Q1 = X_train.quantile(0.25)
        Q3 = X_train.quantile(0.75)
        IQR = Q3 - Q1
        
        # Define outlier boundaries
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Remove outliers from training
        mask_train = ~((X_train < lower_bound) | (X_train > upper_bound)).any(axis=1)
        X_train_clean = X_train[mask_train]
        y_train_clean = y_train[mask_train]
        
        # Remove outliers from test using same boundaries
        mask_test = ~((X_test < lower_bound) | (X_test > upper_bound)).any(axis=1)
        X_test_clean = X_test[mask_test]
        y_test_clean = y_test[mask_test]
        
    elif mode == 'train_only':
        # Calculate IQR and remove outliers only from training
        Q1 = X_train.quantile(0.25)
        Q3 = X_train.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Remove outliers only from training
        mask_train = ~((X_train < lower_bound) | (X_train > upper_bound)).any(axis=1)
        X_train_clean = X_train[mask_train]
        y_train_clean = y_train[mask_train]
        
        # Test data remains unchanged
        X_test_clean = X_test.copy()
        y_test_clean = y_test.copy()
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_clean, y_train_clean, X_test_clean, y_test_clean


# =============================================================================
# 5-FOLD CROSS VALIDATION WITH PREPROCESSING
# =============================================================================

def perform_cv_with_preprocessing(df, normalize_mode='train_only', 
                                  missing_mode='train_only', 
                                  outlier_mode='train_only',
                                  n_splits=5):
    """
    Perform 5-fold cross validation with configurable preprocessing
    
    Parameters:
    -----------
    df : DataFrame - Full dataset
    normalize_mode : str - 'all' or 'train_only' for normalization
    missing_mode : str - 'all' or 'train_only' for missing value replacement
    outlier_mode : str - 'all' or 'train_only' for outlier removal
    n_splits : int - Number of CV folds
    
    Returns:
    --------
    results : dict - Results for each model
    """
    
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    
    # Initialize models
    models = {
        'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
    }
    
    # Store results
    results = {name: {'accuracy': [], 'f1': [], 'auc': [], 'precision': [], 'recall': []} for name in models.keys()}
    
    # 5-Fold Cross Validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    print(f"\n{'='*80}")
    print(f"Configuration: Normalize={normalize_mode}, Missing={missing_mode}, Outlier={outlier_mode}")
    print(f"{'='*80}\n")
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"Fold {fold}/{n_splits}")
        print("-" * 40)
        
        # Split data
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
        
        print(f"Original - Train: {X_train.shape}, Test: {X_test.shape}")
        
        # Apply preprocessing functions
        # 1. Missing values replacement
        X_train, y_train, X_test, y_test = replace_missing_values(
            X_train, y_train, X_test, y_test, mode=missing_mode
        )
        print(f"After missing value replacement - Train: {X_train.shape}, Test: {X_test.shape}")
        
        # 2. Outlier removal
        X_train, y_train, X_test, y_test = remove_outliers(
            X_train, y_train, X_test, y_test, mode=outlier_mode
        )
        print(f"After outlier removal - Train: {X_train.shape}, Test: {X_test.shape}")
        
        # 3. Normalization
        X_train_norm, X_test_norm = normalize_data(
            X_train, X_test, mode=normalize_mode
        )
        
        # Train and evaluate each model
        for model_name, model in models.items():
            # Train
            model.fit(X_train_norm, y_train)
            
            # Predict
            y_pred = model.predict(X_test_norm)
            y_pred_proba = model.predict_proba(X_test_norm)[:, 1]
            
            # Calculate metrics
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_pred_proba)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            
            # Store results
            results[model_name]['accuracy'].append(acc)
            results[model_name]['f1'].append(f1)
            results[model_name]['auc'].append(auc)
            results[model_name]['precision'].append(precision)
            results[model_name]['recall'].append(recall)
            
            print(f"  {model_name:15s} - Acc: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}")
        
        print()
    
    # Print average results
    print(f"\n{'='*80}")
    print("AVERAGE RESULTS ACROSS 5 FOLDS")
    print(f"{'='*80}")
    print(f"Configuration: Normalize={normalize_mode}, Missing={missing_mode}, Outlier={outlier_mode}\n")
    
    for model_name in models.keys():
        avg_acc = np.mean(results[model_name]['accuracy'])
        avg_f1 = np.mean(results[model_name]['f1'])
        avg_auc = np.mean(results[model_name]['auc'])
        avg_precision = np.mean(results[model_name]['precision'])
        avg_recall = np.mean(results[model_name]['recall'])
        
        std_acc = np.std(results[model_name]['accuracy'])
        std_f1 = np.std(results[model_name]['f1'])
        std_auc = np.std(results[model_name]['auc'])
        std_precision = np.std(results[model_name]['precision'])
        std_recall = np.std(results[model_name]['recall'])
        
        print(f"{model_name}:")
        print(f"  Accuracy:  {avg_acc:.4f} (±{std_acc:.4f})")
        print(f"  Precision: {avg_precision:.4f} (±{std_precision:.4f})")
        print(f"  Recall:    {avg_recall:.4f} (±{std_recall:.4f})")
        print(f"  F1 Score:  {avg_f1:.4f} (±{std_f1:.4f})")
        print(f"  AUC-ROC:   {avg_auc:.4f} (±{std_auc:.4f})")
        print()
    
    return results


# =============================================================================
# RUN EXPERIMENTS
# =============================================================================


# Example 2: All preprocessing on both train and test
print("\n" + "="*80)
print("EXPERIMENT 2: All preprocessing on BOTH TRAIN AND TEST")
print("="*80)
results_all = perform_cv_with_preprocessing(
    df,
    normalize_mode='all',
    missing_mode='all',
    outlier_mode='all'
)



Dataset shape: (768, 9)

First few rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               76

In [15]:
results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='all',
    missing_mode='all',
    outlier_mode='train_only'
)


Configuration: Normalize=all, Missing=all, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
  CatBoost        - Acc: 0.7727, F1: 0.6067, AUC: 0.9041, Prec: 0.7714, Rec: 0.5000
  RandomForest    - Acc: 0.7857, F1: 0.6452, AUC: 0.8932, Prec: 0.7692, Rec: 0.5556
  XGBoost         - Acc: 0.7597, F1: 0.5747, AUC: 0.8685, Prec: 0.7576, Rec: 0.4630

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490, 8), Test: (154, 8)
  CatBoost        - Acc: 0.8701, F1: 0.7959, AUC: 0.9250, Prec: 0.8864, Rec: 0.7222
  RandomForest    - Acc: 0.8636, F1: 0.7742, AUC: 0.9297, Prec: 0.9231, Rec: 0.6667
  XGBoost         - Acc: 0.8377, F1: 0.7368, AUC: 0.9272, Prec: 0.8537, Rec: 0.6481



In [16]:

results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='all',
    missing_mode='train_only',
    outlier_mode='train_only'
)


Configuration: Normalize=all, Missing=train_only, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
  CatBoost        - Acc: 0.7273, F1: 0.5532, AUC: 0.8069, Prec: 0.6500, Rec: 0.4815
  RandomForest    - Acc: 0.7273, F1: 0.5435, AUC: 0.7848, Prec: 0.6579, Rec: 0.4630
  XGBoost         - Acc: 0.6558, F1: 0.5047, AUC: 0.7611, Prec: 0.5094, Rec: 0.5000

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490, 8), Test: (154, 8)
  CatBoost        - Acc: 0.7143, F1: 0.5000, AUC: 0.7470, Prec: 0.6471, Rec: 0.4074
  RandomForest    - Acc: 0.7273, F1: 0.5116, AUC: 0.7755, Prec: 0.6875, Rec: 0.4074
  XGBoost         - Acc: 0.7273, F1: 0.5962, AUC: 0.7794, Prec: 0.6200, Rec: 0

In [17]:

results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='train_only',
    missing_mode='train_only',
    outlier_mode='train_only'
)


Configuration: Normalize=train_only, Missing=train_only, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
  CatBoost        - Acc: 0.6948, F1: 0.4198, AUC: 0.7922, Prec: 0.6296, Rec: 0.3148
  RandomForest    - Acc: 0.7013, F1: 0.4524, AUC: 0.7770, Prec: 0.6333, Rec: 0.3519
  XGBoost         - Acc: 0.7403, F1: 0.5652, AUC: 0.7946, Prec: 0.6842, Rec: 0.4815

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490, 8), Test: (154, 8)
  CatBoost        - Acc: 0.7208, F1: 0.4110, AUC: 0.7869, Prec: 0.7895, Rec: 0.2778
  RandomForest    - Acc: 0.7468, F1: 0.5185, AUC: 0.8074, Prec: 0.7778, Rec: 0.3889
  XGBoost         - Acc: 0.7403, F1: 0.5122, AUC: 0.7919, Prec: 0.7500,